# PAEC Full Evaluation Suite

Runs all 6 experiments from the design spec:
1. Main result (CoordinationQA)
2. Explicit-to-applied gap (SimpleToM)
3. ToM accuracy (ToMi)
4. Calibration (ECE, Brier)
5. Prefix ablation
6. Vacuity analysis + case studies

In [ ]:
!pip install -q torch transformers peft datasets bitsandbytes accelerate tqdm pyyaml scipy scikit-learn
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/vualidon/tom_agent_collab.git /content/tom_agent_collab 2>/dev/null || (cd /content/tom_agent_collab && git pull)
!git clone https://github.com/facebookresearch/ToMi.git /content/tom_agent_collab/external/ToMi 2>/dev/null || echo 'Already cloned'
!git clone https://github.com/eric-ai-lab/llm_coordination.git /content/tom_agent_collab/external/llm_coordination 2>/dev/null || echo 'Already cloned'
%cd /content/tom_agent_collab

In [ ]:
import torch
import numpy as np
import json
from datetime import datetime
from src.config import PAECConfig
from src.models.model_loader import load_base_model, load_model_with_prefix
from src.inference.perspective_runner import PerspectiveRunner
from src.baselines.standard_prompting import predict_standard
from src.baselines.simtom_prompting import predict_simtom
from src.baselines.self_consistency import predict_self_consistency
from src.evaluation.simpletom_eval import load_simpletom, evaluate_simpletom
from src.evaluation.tomi_eval import evaluate_tomi
from src.evaluation.coordination_qa import load_coordination_qa, evaluate_coordination_qa
from src.evaluation.calibration import expected_calibration_error, brier_score, mcnemar_test, bootstrap_ci, accuracy_when_confident
from src.data.prefix_training_data import combine_training_data
from transformers import set_seed as hf_set_seed

config = PAECConfig.from_yaml('configs/default.yaml')
runner = PerspectiveRunner(config)

def set_all_seeds(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    hf_set_seed(seed)
    torch.backends.cudnn.deterministic = True

In [ ]:
# Load all datasets
simpletom_data = load_simpletom()
print(f'SimpleToM: {len(simpletom_data)} examples')

# Generate ToMi data
import subprocess
subprocess.run(['python', 'main.py'], cwd='external/ToMi', capture_output=True)
_, tomi_test = combine_training_data('external/ToMi', seed=42)
print(f'ToMi test: {len(tomi_test)} examples')

cqa_data = load_coordination_qa('external/llm_coordination')
print(f'CoordinationQA: {len(cqa_data)} examples')

In [ ]:
# Run all methods across all seeds
import gc
all_results = {}

for seed in config.seeds:
    set_all_seeds(seed)
    print(f'\n{"="*60}\nSeed: {seed}\n{"="*60}')
    seed_results = {}

    # Load models
    base_model, tokenizer = load_base_model(config)

    # --- Experiment 2: SimpleToM ---
    print('\n--- Experiment 2: SimpleToM ---')

    # Baseline 1: Standard
    def std_fn(s, q):
        idx, c = predict_standard(base_model, tokenizer, s, q, ['yes', 'no'])
        return ['yes', 'no'][idx], c
    seed_results['std_simpletom'] = evaluate_simpletom(simpletom_data, std_fn)
    print(f'  Standard: {seed_results["std_simpletom"]["overall_accuracy"]:.3f}')

    # Baseline 2: SimToM
    def sim_fn(s, q):
        idx, c = predict_simtom(base_model, tokenizer, s, q, ['yes', 'no'])
        return ['yes', 'no'][idx], c
    seed_results['sim_simpletom'] = evaluate_simpletom(simpletom_data, sim_fn)
    print(f'  SimToM: {seed_results["sim_simpletom"]["overall_accuracy"]:.3f}')

    # Baseline 3: SC-2
    def sc2_fn(s, q):
        idx, c = predict_self_consistency(base_model, tokenizer, s, q, ['yes', 'no'], n_samples=2, temperature=config.sc_temperature)
        return ['yes', 'no'][idx], c
    seed_results['sc2_simpletom'] = evaluate_simpletom(simpletom_data, sc2_fn)
    print(f'  SC-2: {seed_results["sc2_simpletom"]["overall_accuracy"]:.3f}')

    # Baseline 4: SC-8
    def sc8_fn(s, q):
        idx, c = predict_self_consistency(base_model, tokenizer, s, q, ['yes', 'no'], n_samples=8, temperature=config.sc_temperature)
        return ['yes', 'no'][idx], c
    seed_results['sc8_simpletom'] = evaluate_simpletom(simpletom_data, sc8_fn)
    print(f'  SC-8: {seed_results["sc8_simpletom"]["overall_accuracy"]:.3f}')

    # Baseline 5 / PAEC prompt-only
    def paec_p_fn(s, q):
        r = runner.predict(base_model, base_model, tokenizer, s, q, ['yes', 'no'], use_prompt_perspective=True)
        return ['yes', 'no'][r.answer_idx], r.fused.confidence
    seed_results['paec_prompt_simpletom'] = evaluate_simpletom(simpletom_data, paec_p_fn)
    print(f'  PAEC(prompt): {seed_results["paec_prompt_simpletom"]["overall_accuracy"]:.3f}')

    # PAEC prefix
    del base_model; gc.collect(); torch.cuda.empty_cache()
    model_self, tokenizer = load_model_with_prefix(config, '/content/drive/MyDrive/paec_checkpoints/prefix_self')
    model_partner, _ = load_model_with_prefix(config, '/content/drive/MyDrive/paec_checkpoints/prefix_partner')

    def paec_pfx_fn(s, q):
        r = runner.predict(model_self, model_partner, tokenizer, s, q, ['yes', 'no'])
        return ['yes', 'no'][r.answer_idx], r.fused.confidence
    seed_results['paec_prefix_simpletom'] = evaluate_simpletom(simpletom_data, paec_pfx_fn)
    print(f'  PAEC(prefix): {seed_results["paec_prefix_simpletom"]["overall_accuracy"]:.3f}')

    # --- Experiment 3: ToMi ---
    print('\n--- Experiment 3: ToMi ---')
    def paec_pfx_tomi(s, q):
        r = runner.predict(model_self, model_partner, tokenizer, s, q, ['yes', 'no'])
        return ['yes', 'no'][r.answer_idx], r.fused.confidence
    seed_results['paec_prefix_tomi'] = evaluate_tomi(tomi_test, paec_pfx_tomi)
    print(f'  PAEC(prefix) ToMi: {seed_results["paec_prefix_tomi"]["overall_accuracy"]:.3f}')

    # Cleanup
    del model_self, model_partner; gc.collect(); torch.cuda.empty_cache()
    all_results[seed] = seed_results

print('\n=== All seeds complete ===')

In [ ]:
# Experiment 4: Calibration
print('=== Experiment 4: Calibration ===')
for method in ['paec_prefix_simpletom', 'sc2_simpletom', 'sc8_simpletom']:
    confs = []
    corrs = []
    for seed in config.seeds:
        r = all_results[seed].get(method, {})
        if 'all_confidences' in r:
            confs.append(r['all_confidences'])
            corrs.append(r['all_correctness'])
    if confs:
        c = np.concatenate(confs)
        cr = np.concatenate(corrs)
        ece = expected_calibration_error(c, cr)
        bs = brier_score(c, cr)
        print(f'{method}: ECE={ece:.4f}, Brier={bs:.4f}')

In [ ]:
# Experiment 5: Prefix Ablation
print('=== Experiment 5: Prefix Ablation ===')
for method in ['std_simpletom', 'paec_prompt_simpletom', 'paec_prefix_simpletom']:
    accs = [all_results[s].get(method, {}).get('overall_accuracy', 0) for s in config.seeds]
    print(f'{method}: {np.mean(accs):.3f} +/- {np.std(accs):.3f}')

In [ ]:
# Experiment 6: Vacuity Analysis
print('=== Experiment 6: Vacuity Analysis ===')
# Accuracy-when-confident curve for PAEC prefix on SimpleToM
r = all_results[config.seeds[0]].get('paec_prefix_simpletom', {})
if 'all_confidences' in r:
    vacuity = 1 - r['all_confidences']
    thresholds, accs, covs = accuracy_when_confident(
        r['all_confidences'], r['all_correctness'], vacuity
    )
    print('Vacuity threshold | Accuracy | Coverage')
    for t, a, c in zip(thresholds[::4], accs[::4], covs[::4]):
        print(f'  {t:.2f}            | {a:.3f}    | {c:.3f}')

In [ ]:
# Summary table
print('\n' + '='*80)
print('RESULTS SUMMARY')
print('='*80)
print(f'{"Method":<25} {"SimpleToM":>10} {"ToMi":>10}')
print('-'*50)

methods = ['std', 'sim', 'sc2', 'sc8', 'paec_prompt', 'paec_prefix']
labels = ['Standard', 'SimToM', 'SC-2', 'SC-8', 'PAEC(prompt)', 'PAEC(prefix)']

for method, label in zip(methods, labels):
    st_key = f'{method}_simpletom'
    tm_key = f'{method}_tomi'
    st_accs = [all_results[s].get(st_key, {}).get('overall_accuracy', float('nan')) for s in config.seeds]
    tm_accs = [all_results[s].get(tm_key, {}).get('overall_accuracy', float('nan')) for s in config.seeds]
    st = f'{np.nanmean(st_accs):.3f}' if not all(np.isnan(st_accs)) else 'N/A'
    tm = f'{np.nanmean(tm_accs):.3f}' if not all(np.isnan(tm_accs)) else 'N/A'
    print(f'{label:<25} {st:>10} {tm:>10}')

# Save results
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
def convert(obj):
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, np.floating): return float(obj)
    return obj

with open(f'/content/drive/MyDrive/paec_checkpoints/results_{timestamp}.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=convert)
print(f'\nResults saved to Drive')